<a href="https://colab.research.google.com/github/dasanchez28/03MAIR---Algoritmos-de-Optimizacion/blob/main/TrabajoPractico/Trabajo_Pr%C3%A1ctico_Algoritmos_Diego_Aguado_S%C3%A1nchez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Diego Aguado Sánchez  <br>
Url: https://github.com/dasanchez28/03MAIR---Algoritmos-de-Optimizacion/blob/main/TrabajoPractico/Trabajo_Pr%C3%A1ctico_Algoritmos_Diego_Aguado_S%C3%A1nchez.ipynb<br>
Google Colab: https://colab.research.google.com/drive/13PYge_4TFcSVs_ryPpyaw_ZqowD5hbco?usp=sharing <br>
Problema:
>2. Organizar los horarios de partidos de una jornada de La Liga<br>







                                        

## Descripción del problema

Desde La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma que **maximice la audiencia**.

Los **Horarios disponibles** se conocen a priori y son los siguientes:

| Día | Horas disponibles |
|-----|-------------------|
| Viernes | 20h |
| Sábado | 12h, 16h, 18h, 20h |
| Domingo | 12h, 16h, 18h, 20h |
| Lunes | 20h |


**Categorías de equipos** (según número de seguidores):
- **Categoría A:** 3 equipos (mayor audiencia)
- **Categoría B:** 11 equipos
- **Categoría C:** 6 equipos

**Audiencia base por enfrentamiento** (en S20h, el mejor horario):

| | Cat A | Cat B | Cat C |
|--|--|--|--|
| **Cat A** | 2.0 M | 1.3 M | 1.0 M |
| **Cat B** | 1.3 M | 0.9 M | 0.75 M |
| **Cat C** | 1.0 M | 0.75 M | 0.47 M |

**Coeficientes de reducción por horario:**

| | Viernes | Sábado | Domingo | Lunes |
|--|--|--|--|--|
| **12h** | — | 0.55 | 0.45 | — |
| **16h** | — | 0.70 | 0.75 | — |
| **18h** | — | 0.80 | 0.85 | — |
| **20h** | 0.40 | 1.00 | 1.00 | 0.40 |

**Corrección por coincidencia** (varios partidos en el mismo horario):


| Coincidencias | Partidos simultáneos | Reducción | Factor corrección |
|--|--|--|--|
| 0 | 1 | 0% | 1.00 |
| 1 | 2 | 25% | 0.75 |
| 2 | 3 | 45% | 0.55 |
| 3 | 4 | 60% | 0.40 |
| 4 | 5 | 70% | 0.30 |
| 5 | 6 | 75% | 0.25 |
| 6 | 7 | 78% | 0.22 |
| 7 | 8 | 80% | 0.20 |
| 8 | 9 | 80% | 0.20 |

**Restricciones obligatorias:**
- Debe haber **al menos 1 partido el viernes** y **al menos 1 partido el lunes**.
- Pueden coincidir varios partidos en el mismo slot (incluyendo viernes y lunes).


## Librerías y datos del problema

In [ ]:
import random
import math
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

random.seed(42)
np.random.seed(42)

# Horarios y coeficientes
SLOTS = {
    'V20': 0.40,  # Viernes 20h
    'S12': 0.55,  # Sábado 12h
    'S16': 0.70,  # Sábado 16h
    'S18': 0.80,  # Sábado 18h
    'S20': 1.00,  # Sábado 20h
    'D12': 0.45,  # Domingo 12h
    'D16': 0.75,  # Domingo 16h
    'D18': 0.85,  # Domingo 18h
    'D20': 1.00,  # Domingo 20h
    'L20': 0.40,  # Lunes 20h
}

FRIDAY_SLOTS = {'V20'}
MONDAY_SLOTS = {'L20'}
ALL_SLOTS    = set(SLOTS.keys())

# ── Audiencia base (millones) ──
BASE_AUDIENCE = {
    ('A','A'): 2.00,
    ('A','B'): 1.30, ('B','A'): 1.30,
    ('A','C'): 1.00, ('C','A'): 1.00,
    ('B','B'): 0.90,
    ('B','C'): 0.75, ('C','B'): 0.75,
    ('C','C'): 0.47,
}

# ── Factor de corrección por coincidencia (0 a 8 coincidencias = 1 a 9 partidos simultáneos) ──
COINCIDENCE_FACTOR = {
    1: 1.00,  # 0 coincidencias
    2: 0.75,  # 1 coincidencia
    3: 0.55,  # 2 coincidencias
    4: 0.40,  # 3 coincidencias
    5: 0.30,  # 4 coincidencias
    6: 0.25,  # 5 coincidencias
    7: 0.22,  # 6 coincidencias
    8: 0.20,  # 7 coincidencias
    9: 0.20,  # 8 coincidencias
}
def get_cf(n): return COINCIDENCE_FACTOR.get(n, 0.20)

# Jornada del enunciado
matches = [
    ('Celta',      'Real Madrid', 'B', 'A'),
    ('Valencia',   'R. Sociedad', 'B', 'A'),
    ('Mallorca',   'Eibar',       'C', 'C'),
    ('Athletic',   'Barcelona',   'B', 'A'),
    ('Leganés',    'Osasuna',     'C', 'C'),
    ('Villarreal', 'Granada',     'B', 'C'),
    ('Alavés',     'Levante',     'B', 'B'),
    ('Espanyol',   'Sevilla',     'B', 'B'),
    ('Betis',      'Valladolid',  'B', 'C'),
    ('Atlético',   'Getafe',      'B', 'B'),
]

print(f"✅ {len(SLOTS)} horarios disponibles | {len(matches)} partidos en la jornada")


✅ 10 horarios disponibles | 10 partidos en la jornada


---
# Modelo

## ¿Cómo represento el espacio de soluciones?


### Respuesta

Una **solución** es una lista de 10 elementos donde la posición `i` contiene el horario asignado al partido `i`:

```
solución = ['V20', 'S12', 'S16', 'S18', 'S20', 'D16', 'V20', 'D18', 'D20', 'L20']
             ↑       ↑                            ↑                          ↑
          partido 0  partido 1  ...     partido 6 (2º en V20)           partido 9
```

Cada elemento pertenece al conjunto de 10 horarios posibles: `{V20, S12, S16, S18, S20, D12, D16, D18, D20, L20}`.

Cualquier partido puede asignarse a cualquier slot, incluidos `V20` y `L20`. La única restricción es que al menos uno de los 10 partidos esté en `V20` y al menos uno en `L20`.


## ¿Cuál es la función objetivo?

### Respuesta

Se busca **maximizar la audiencia total** de la jornada:

$$\text{Maximizar} \quad F(s) = \sum_{i=1}^{10} \text{Base}(c_i^{local}, c_i^{visit}) \times \text{Coef}(s_i) \times \text{CF}(n_{s_i})$$

Donde:
- $\text{Base}(c_i^{local}, c_i^{visit})$ = audiencia base del partido según categorías de los equipos (M espectadores en S20h)
- $\text{Coef}(s_i)$ = coeficiente del horario asignado $s_i$ respecto al máximo (S20h = 1.0)
- $\text{CF}(n_{s_i})$ = factor de corrección por coincidencia, donde $n_{s_i}$ es el número de partidos asignados al mismo slot $s_i$


In [ ]:
def evaluate(assignment):
    """Calcula la audiencia total (función objetivo a maximizar)."""
    slot_count = defaultdict(int)
    for slot in assignment:
        slot_count[slot] += 1

    total = 0.0
    for i, (_, _, c1, c2) in enumerate(matches):
        base  = BASE_AUDIENCE[(c1, c2)]
        coef  = SLOTS[assignment[i]]
        cf    = get_cf(slot_count[assignment[i]])
        total += base * coef * cf
    return total

# Demostración con la solución del enunciado
example = ['V20','S12','S16','S18','S20','D16','D16','D18','D20','L20']
print(f"Audiencia total (jornada ejemplo): {evaluate(example):.4f} M")


Audiencia total (jornada ejemplo): 5.8771 M


## ¿Cómo implemento las restricciones?

### Respuesta

El problema tiene **restricciones duras** (hard constraints) que toda solución válida debe satisfacer:

1. **Al menos 1 partido el viernes** → el slot `V20` aparece **una o más veces** en la lista
2. **Al menos 1 partido el lunes** → el slot `L20` aparece **una o más veces** en la lista

El resto de partidos puede distribuirse libremente entre cualquier slot, **incluyendo `V20` y `L20`**.

Se implementan mediante:
- Una **función de validación** que comprueba que exista al menos un V20 y un L20
- La **construcción de soluciones iniciales** garantiza siempre asignar al menos un partido al viernes y otro al lunes antes de distribuir el resto por todos los slots disponibles

No se penalizan en la función objetivo, sino que se rechazan directamente.


In [ ]:
def is_valid(assignment):
    """
    Verifica las restricciones duras:
      - Al menos 1 partido en Viernes (V20)
      - Al menos 1 partido en Lunes (L20)
    """
    fridays = sum(1 for s in assignment if s in FRIDAY_SLOTS)
    mondays = sum(1 for s in assignment if s in MONDAY_SLOTS)
    return fridays >= 1 and mondays >= 1

print(f"Ejemplo del enunciado válido: {is_valid(example)}")

# Ejemplo de solución inválida (sin viernes)
invalid = ['S20'] * 10
print(f"Todos en S20 → válido: {is_valid(invalid)}")

# Solución con varios partidos en viernes → también válida
multi_friday = ['V20', 'V20', 'S20', 'S18', 'S16', 'D20', 'D18', 'D16', 'S12', 'L20']
print(f"Dos partidos en V20 → válido: {is_valid(multi_friday)}")


Ejemplo del enunciado válido: True
Todos en S20 → válido: False
Dos partidos en V20 → válido: True


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

### Respuesta

#### Espacio de soluciones

Con **10 partidos** y **10 horarios** posibles, sin restricciones el espacio sería:

$$|S_{total}| = 10^{10} = 10{,}000{,}000{,}000 \text{ soluciones}$$

Con las restricciones (**al menos 1 viernes** y **al menos 1 lunes**), aplicamos el principio de inclusión-exclusión:

$$|S_{válidas}| = 10^{10} - |\text{sin V20}| - |\text{sin L20}| + |\text{sin V20 y sin L20}|$$

$$= 10^{10} - 9^{10} - 9^{10} + 8^{10}$$

$$= 10{,}000{,}000{,}000 - 3{,}486{,}784{,}401 - 3{,}486{,}784{,}401 + 1{,}073{,}741{,}824$$

$$\approx 4.1 \times 10^9 \text{ soluciones válidas}$$

#### Orden de complejidad

- **Evaluación de una solución:** $O(n)$ donde $n$ = número de partidos
- **Búsqueda exhaustiva:** $O(k^n)$ con $k=10$ slots → **completamente intratable**
- **Este problema pertenece a la familia de problemas de asignación**, que en su forma general es **NP-difícil**
- La interacción entre coincidencias (la audiencia de un partido depende de cuántos otros comparten su slot) rompe la optimalidad de cualquier enfoque greedy


In [ ]:
from math import comb

n_matches = 10
n_slots   = 10

# Sin restricciones
sin_restricciones = n_slots ** n_matches
print(f"Sin restricciones:                    {sin_restricciones:>15,}")

# Con restricción: al menos 1 V20 y al menos 1 L20 (inclusión-exclusión)
sin_v20         = 9 ** n_matches   # ningún partido en V20
sin_l20         = 9 ** n_matches   # ningún partido en L20
sin_v20_ni_l20  = 8 ** n_matches   # ningún partido en V20 ni L20

con_restricciones = sin_restricciones - sin_v20 - sin_l20 + sin_v20_ni_l20
print(f"Con restricciones (≥1 V20 y ≥1 L20):  {con_restricciones:>15,}")
print(f"  = 10^10 - 9^10 - 9^10 + 8^10")
print(f"  = {sin_restricciones:,} - {sin_v20:,} - {sin_l20:,} + {sin_v20_ni_l20:,}")
print()
print(f"Orden de magnitud: {con_restricciones:.2e} soluciones válidas")


Sin restricciones:                     10,000,000,000
Con restricciones (≥1 V20 y ≥1 L20):    4,100,173,022
  = 10^10 - 9^10 - 9^10 + 8^10
  = 10,000,000,000 - 3,486,784,401 - 3,486,784,401 + 1,073,741,824

Orden de magnitud: 4.10e+09 soluciones válidas


#Diseño
- ¿Que técnica utilizo? ¿Por qué?

### Respuesta

Se utiliza **Simulated Annealing (Recocido Simulado)** como algoritmo principal, precedido de una solución inicial **Greedy**.

#### ¿Por qué Simulated Annealing?

La **función objetivo no es separable**: la audiencia de un partido depende de cuántos otros partidos comparten su horario (factor de coincidencia). Esto hace que los algoritmos greedy queden atrapados en óptimos locales.

SA es ideal porque:
1. **Acepta soluciones peores temporalmente** (con probabilidad $e^{\Delta/T}$), escapando de trampas locales
2. **Converge gradualmente** al reducir la temperatura, comportándose como búsqueda local al final
3. **Implementación sencilla** con solo 2 parámetros a ajustar (T₀ y α)
4. **Garantías empíricas** suficientes para el tamaño del problema (10 partidos)

#### Descripción del algoritmo

```
1. Generar solución inicial s₀ con Greedy
2. T ← T₀
3. Repetir N iteraciones:
   a. Generar vecino s' intercambiando slots de dos partidos aleatorios
   b. Si s' es válido:
      - Calcular Δ = F(s') - F(s)
      - Si Δ > 0: aceptar s' siempre
      - Si Δ ≤ 0: aceptar s' con probabilidad e^(Δ/T)
   c. T ← T × α   (enfriamiento)
4. Devolver mejor solución encontrada
```


 A continuación se definen dos funciones auxiliares:
 - greedy_initial(): construye una solución inicial asignando los partidos a horarios de forma voraz
 - show_assignment(): muestra una tabla detallada con el desglose de audiencia de una asignación dada

In [ ]:
def greedy_initial():
    """
    Construye una solución inicial de forma voraz:
    - Asigna al menos 1 partido al viernes y 1 al lunes (los de menor audiencia base)
    - El resto puede asignarse a CUALQUIER slot (incluyendo V20 y L20)
      eligiendo en cada paso el que maximiza la audiencia marginal
    """
    n = len(matches)
    order = sorted(range(n),
                   key=lambda i: BASE_AUDIENCE[(matches[i][2], matches[i][3])],
                   reverse=True)

    assignment = [None] * n
    slot_count = defaultdict(int)

    # Restricciones obligatorias: al menos 1 viernes y 1 lunes → asignar a los de menor valor
    friday_idx = order[-1]
    assignment[friday_idx] = 'V20'
    slot_count['V20'] += 1

    monday_idx = order[-2]
    assignment[monday_idx] = 'L20'
    slot_count['L20'] += 1

    # Resto: greedy sobre TODOS los slots disponibles (incluyendo V20 y L20)
    for idx in order[:-2]:
        best_slot, best_val = None, -1
        base = BASE_AUDIENCE[(matches[idx][2], matches[idx][3])]
        for sl in ALL_SLOTS:
            val = base * SLOTS[sl] * get_cf(slot_count[sl] + 1)
            if val > best_val:
                best_val, best_slot = val, sl
        assignment[idx] = best_slot
        slot_count[best_slot] += 1

    return assignment


def show_assignment(assignment, title="Asignación"):
    """Muestra una tabla detallada de la asignación."""
    slot_count = defaultdict(int)
    for s in assignment: slot_count[s] += 1

    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")
    print(f"{'Partido':<28} {'Cat':>4} {'Slot':>4} {'Base':>5} {'Coef':>5} {'B×C':>6} {'CF':>5} {'Final':>6}")
    print("-"*70)
    total = 0.0
    for i, (h1, h2, c1, c2) in enumerate(matches):
        slot  = assignment[i]
        base  = BASE_AUDIENCE[(c1, c2)]
        coef  = SLOTS[slot]
        n     = slot_count[slot]
        cf    = get_cf(n)
        final = base * coef * cf
        total += final
        flag  = " ⚠" if n > 1 else ""
        print(f"{h1+' – '+h2:<28} {c1+'-'+c2:>4} {slot:>4} "
              f"{base:>5.2f} {coef:>5.2f} {base*coef:>6.3f} {cf:>5.2f} {final:>6.3f}{flag}")
    print("-"*70)
    print(f"{'AUDIENCIA TOTAL':>62} {total:>6.3f} M")
    print(f"  ⚠ = Coincidencia de horario")
    return total


greedy = greedy_initial()
score_greedy = show_assignment(greedy, "Solución Greedy")
score_example = evaluate(example)
print(f"\nMejora vs ejemplo enunciado: +{score_greedy - score_example:.4f} M")



  Solución Greedy
Partido                       Cat Slot  Base  Coef    B×C    CF  Final
----------------------------------------------------------------------
Celta – Real Madrid           B-A  D20  1.30  1.00  1.300  0.75  0.975 ⚠
Valencia – R. Sociedad        B-A  S20  1.30  1.00  1.300  0.75  0.975 ⚠
Mallorca – Eibar              C-C  L20  0.47  0.40  0.188  1.00  0.188
Athletic – Barcelona          B-A  D18  1.30  0.85  1.105  1.00  1.105
Leganés – Osasuna             C-C  V20  0.47  0.40  0.188  1.00  0.188
Villarreal – Granada          B-C  S20  0.75  1.00  0.750  0.75  0.562 ⚠
Alavés – Levante              B-B  S18  0.90  0.80  0.720  1.00  0.720
Espanyol – Sevilla            B-B  D16  0.90  0.75  0.675  1.00  0.675
Betis – Valladolid            B-C  S16  0.75  0.70  0.525  1.00  0.525
Atlético – Getafe             B-B  D20  0.90  1.00  0.900  0.75  0.675 ⚠
----------------------------------------------------------------------
                                               AUD

## Implementación: Simulated Annealing:

Y ahora se define el algoritmo principal de optimización.
La función `simulated_annealing()` parte de la solución greedy y explora el espacio de soluciones
aceptando también movimientos que empeoran el resultado con cierta probabilidad,
lo que le permite escapar de óptimos locales y encontrar soluciones de mayor calidad.

In [ ]:
def simulated_annealing(n_iter=200_000, T0=1.0, cooling=0.9999, seed=42):
    """
    Optimiza la asignación mediante Simulated Annealing.

    Parámetros:
        n_iter   : número de iteraciones
        T0       : temperatura inicial
        cooling  : factor de enfriamiento multiplicativo
        seed     : semilla aleatoria

    Retorna:
        best_assignment  : mejor lista de horarios encontrada
        best_score       : audiencia total de la mejor solución
        history          : lista de (iteración, mejor, actual, temperatura)
    """
    random.seed(seed)
    current     = greedy_initial()
    current_score = evaluate(current)
    best        = current[:]
    best_score  = current_score
    history     = []
    T           = T0
    n           = len(matches)

    for it in range(n_iter):
        # Vecino: intercambiar slots de dos partidos aleatorios
        i, j    = random.sample(range(n), 2)
        neighbor = current[:]
        neighbor[i], neighbor[j] = neighbor[j], neighbor[i]

        if not is_valid(neighbor):
            T *= cooling
            continue

        ns    = evaluate(neighbor)
        delta = ns - current_score

        # Criterio de aceptación de Metropolis
        if delta > 0 or random.random() < math.exp(delta / T):
            current       = neighbor
            current_score = ns
            if current_score > best_score:
                best       = current[:]
                best_score = current_score

        T *= cooling

        if it % 10_000 == 0:
            history.append((it, best_score, current_score, T))

    return best, best_score, history


print("Ejecutando Simulated Annealing (200 000 iteraciones)...")
best_assign, best_score, history = simulated_annealing()
print(f"Completado. Audiencia óptima: {best_score:.4f} M")


Ejecutando Simulated Annealing (200 000 iteraciones)...
Completado. Audiencia óptima: 6.6085 M


## Resultados
A continuación se muestran los resultados obtenidos tras ejecutar ambos algoritmos sobre la jornada de ejemplo.
Se compara la audiencia total conseguida por la asignación del enunciado, la solución greedy y la solución
optimizada mediante Simulated Annealing, con el objetivo de cuantificar la mejora obtenida respecto a la línea base.

In [ ]:
score_opt = show_assignment(best_assign, "Solución Óptima — Simulated Annealing")

print("\n" + "="*52)
print("  RESUMEN COMPARATIVO")
print("="*52)
print(f"  Jornada ejemplo (enunciado):  {score_example:.4f} M")
print(f"  Solución Greedy:              {score_greedy:.4f} M  (+{100*(score_greedy-score_example)/score_example:.1f}%)")
print(f"  Simulated Annealing (óptima): {score_opt:.4f} M  (+{100*(score_opt-score_example)/score_example:.1f}%)")
print("="*52)



  Solución Óptima — Simulated Annealing
Partido                       Cat Slot  Base  Coef    B×C    CF  Final
----------------------------------------------------------------------
Celta – Real Madrid           B-A  D18  1.30  0.85  1.105  1.00  1.105
Valencia – R. Sociedad        B-A  S20  1.30  1.00  1.300  0.75  0.975 ⚠
Mallorca – Eibar              C-C  L20  0.47  0.40  0.188  1.00  0.188
Athletic – Barcelona          B-A  S18  1.30  0.80  1.040  1.00  1.040
Leganés – Osasuna             C-C  V20  0.47  0.40  0.188  1.00  0.188
Villarreal – Granada          B-C  S16  0.75  0.70  0.525  1.00  0.525
Alavés – Levante              B-B  D16  0.90  0.75  0.675  1.00  0.675
Espanyol – Sevilla            B-B  D20  0.90  1.00  0.900  0.75  0.675 ⚠
Betis – Valladolid            B-C  D20  0.75  1.00  0.750  0.75  0.562 ⚠
Atlético – Getafe             B-B  S20  0.90  1.00  0.900  0.75  0.675 ⚠
----------------------------------------------------------------------
                            

## Análisis de la Solución Óptima

### Estrategia encontrada por el algoritmo

- **Partidos C-C → Viernes y Lunes:** absorben los slots obligatorios de menor coeficiente (0.40),
  liberando los mejores horarios del fin de semana para los partidos de mayor valor.

- **Partidos B-A en slots distintos sin coincidir:** jugar solo en un horario intermedio genera más
  audiencia que compartir el mejor slot: 1.30 × 0.85 × 1.00 = 1.105M frente a 1.30 × 1.00 × 0.75 = 0.975M.

- **Coincidir en S20 solo cuando vale la pena:** la coincidencia es rentable únicamente si la audiencia
  base es suficientemente alta para compensar la penalización del 25%, por ejemplo:
  0.90 × 1.00 × 0.75 = 0.675M > 0.90 × 0.70 × 1.00 = 0.630M.

- **Slots intermedios (S16, D16) para partidos medios:** evitan saturar los horarios premium y
  reducir la audiencia de los partidos de mayor atractivo.

### Comparativa final

In [ ]:
print(f"{'Solución':<35} {'Audiencia':>10}  {'vs Ejemplo':>10}")
print("-"*58)
print(f"{'Jornada ejemplo (enunciado)':<35} {score_example:>10.4f}M  {'—':>10}")
print(f"{'Solución Greedy':<35} {score_greedy:>10.4f}M  {'+'+str(round(score_greedy-score_example,4)):>10}M")
print(f"{'Simulated Annealing (óptima)':<35} {score_opt:>10.4f}M  {'+'+str(round(score_opt-score_example,4)):>10}M")
print()
print(f"Mejora total SA vs ejemplo: +{100*(score_opt-score_example)/score_example:.1f}%")
print()
print("Asignación óptima final:")
for i, (h1, h2, _, _) in enumerate(matches):
    marker = " ← V obligatorio" if best_assign[i]=='V20' else " ← L obligatorio" if best_assign[i]=='L20' else ""
    print(f"  {h1:<12} – {h2:<12} → {best_assign[i]}{marker}")


Solución                             Audiencia  vs Ejemplo
----------------------------------------------------------
Jornada ejemplo (enunciado)             5.8771M           —
Solución Greedy                         6.5885M     +0.7114M
Simulated Annealing (óptima)            6.6085M     +0.7314M

Mejora total SA vs ejemplo: +12.4%

Asignación óptima final:
  Celta        – Real Madrid  → D18
  Valencia     – R. Sociedad  → S20
  Mallorca     – Eibar        → L20 ← L obligatorio
  Athletic     – Barcelona    → S18
  Leganés      – Osasuna      → V20 ← V obligatorio
  Villarreal   – Granada      → S16
  Alavés       – Levante      → D16
  Espanyol     – Sevilla      → D20
  Betis        – Valladolid   → D20
  Atlético     – Getafe       → S20


El algoritmo de Simulated Annealing consigue una audiencia total de **6.6085M**, lo que supone una mejora
del **+12.4%** respecto a la jornada de ejemplo del enunciado (5.8771M) y una ligera mejora sobre la
solución greedy (6.5885M). Esto demuestra que la metaheurística es capaz de escapar del óptimo local
en el que queda atrapado el greedy, encontrando una asignación de mayor calidad. La clave de la solución
óptima reside en reservar los slots premium del fin de semana para los partidos de mayor audiencia base,
mientras que los partidos de menor valor absorben los horarios obligatorios de viernes y lunes.


 # Nota sobre el uso de IA generativa
 Durante el desarrollo de este trabajo se ha utilizado IA generativa como apoyo en tareas de formato y presentación: estructuración del enunciado en tablas y maquetación general del notebook.